# 02 · Filtering rows (WHERE)

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~30 min**

## Learning objectives
- filter rows with `WHERE` and comparisons (`=`, `<`, `>`, `<>`);
- combine conditions with `AND`, `OR`, `NOT`;
- use `IN`, `BETWEEN`, `LIKE` and `IS NULL`.

`WHERE` keeps only the rows that match a condition. It is the workhorse of every query.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. A simple condition

In [ ]:
%%sql
SELECT * FROM samples WHERE environment = 'Soil';

The opposite test uses `<>` (not equal). Here, every sample that is not Soil.

In [ ]:
%%sql
-- <> means 'not equal to'
SELECT sample_id, environment
FROM samples
WHERE environment <> 'Soil';

## 2. Numeric comparison

In [ ]:
%%sql
SELECT sample_id, ph FROM samples WHERE ph < 6.5;

## 3. Combine with AND / OR

In [ ]:
%%sql
SELECT sample_id, environment, treatment
FROM samples
WHERE environment = 'Freshwater' AND treatment = 'Amended';

`OR` widens the net; wrap `OR` in parentheses when you mix it with `AND` so the logic is unambiguous.

In [ ]:
%%sql
-- amended samples that are EITHER acidic OR warm
SELECT sample_id, environment, ph, temperature_c
FROM samples
WHERE treatment = 'Amended'
  AND (ph < 6.5 OR temperature_c > 18);

## 4. IN — match a list of values

In [ ]:
%%sql
SELECT genus, phylum FROM taxa
WHERE phylum IN ('Cyanobacteriota', 'Euryarchaeota');

`NOT IN` keeps everything except a list, useful to exclude a few known groups.

In [ ]:
%%sql
-- all taxa EXCEPT the two archaeal/photosynthetic phyla above
SELECT genus, phylum
FROM taxa
WHERE phylum NOT IN ('Cyanobacteriota', 'Euryarchaeota')
LIMIT 10;

## 5. BETWEEN — an inclusive range

In [ ]:
%%sql
SELECT sample_id, ph FROM samples WHERE ph BETWEEN 6.5 AND 7.5;

## 6. LIKE — text pattern (`%` = any characters)

In [ ]:
%%sql
SELECT genus, phylum FROM taxa WHERE genus LIKE 'B%';

`%` matches any run of characters; `_` matches exactly one. You can also anchor the pattern at the end.

In [ ]:
%%sql
-- families ending in 'aceae' (the standard bacterial family suffix)
SELECT DISTINCT family
FROM taxa
WHERE family LIKE '%aceae';

## 7. IS NULL — missing values
Some samples have no recorded pH. You must use `IS NULL`, never `= NULL`.

In [ ]:
%%sql
SELECT sample_id, environment, ph FROM samples WHERE ph IS NULL;

`IS NOT NULL` is the mirror image: keep only rows that do have a value. Combine it with another condition freely.

In [ ]:
%%sql
-- samples that HAVE a pH and are on the acidic side
SELECT sample_id, ph
FROM samples
WHERE ph IS NOT NULL AND ph < 6.5;

---
## Exercises

**Exercise.** Show all samples from the `Amended` treatment.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT * FROM samples WHERE treatment = 'Amended';
```
</details>

**Exercise.** List the genus and family of all taxa in the phylum `Bacillota`.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT genus, family FROM taxa WHERE phylum = 'Bacillota';
```
</details>

**Exercise.** Find samples warmer than 20 °C (`temperature_c`).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, temperature_c FROM samples WHERE temperature_c > 20;
```
</details>

**Exercise.** Find the samples whose `depth_cm` is missing.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id FROM samples WHERE depth_cm IS NULL;
```
</details>

### Recap
- `WHERE condition` keeps matching rows; combine with `AND`/`OR`/`NOT`.
- `IN (…)`, `BETWEEN a AND b`, `LIKE 'B%'` are shortcuts.
- Missing values need `IS NULL` / `IS NOT NULL` (never `= NULL`).
- Next: 03 · Sorting, limiting & computed columns.